<a href="https://colab.research.google.com/github/gretadive/correlacion_actividad_solar_elnino/blob/main/notebooks/%2006_integracion_precipitacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook integra la serie mensual de precipitación ERA5 con el dataset previamente construido de ICEN, ONI y SSN.

In [4]:
import os
import subprocess

repo = "correlacion_actividad_solar_elnino"
ruta_repo = f"/content/{repo}"

if not os.path.exists(ruta_repo):
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/gretadive/correlacion_actividad_solar_elnino.git"
        ],
        check=True
    )
    print("✅ Repositorio clonado correctamente")
else:
    print("✅ El repositorio ya está disponible")

✅ Repositorio clonado correctamente


In [11]:
%cd /content/correlacion_actividad_solar_elnino

!git pull origin main

/content/correlacion_actividad_solar_elnino
From https://github.com/gretadive/correlacion_actividad_solar_elnino
 * branch            main       -> FETCH_HEAD
Already up to date.


In [12]:
!ls data/processed

dataset_integrado.csv  oni_procesado.csv		 README.md
icen_procesado.csv     precipitacion.ERA5_procesado.csv  ssn_procesado.csv


In [13]:
import pandas as pd

# Dataset integrado: ICEN + ONI + SSN
dataset = pd.read_csv(
    "data/processed/dataset_integrado.csv",
    parse_dates=["fecha"]
)

# Precipitación ERA5
precipitacion = pd.read_csv(
    "data/processed/precipitacion.ERA5_procesado.csv",
    parse_dates=["fecha"]
)

print("Dataset ICEN-ONI-SSN:", dataset.shape)
print("Precipitación ERA5:", precipitacion.shape)

Dataset ICEN-ONI-SSN: (917, 6)
Precipitación ERA5: (920, 4)


In [14]:
print("Columnas dataset:")
print(dataset.columns.tolist())

print("\nColumnas precipitación:")
print(precipitacion.columns.tolist())

Columnas dataset:
['fecha', 'anio', 'mes', 'ICEN', 'ONI', 'SSN']

Columnas precipitación:
['fecha', 'año', 'mes', 'precipitacion_mm']


In [15]:
precip_merge = precipitacion[
    ["fecha", "precipitacion_mm"]
].copy()

dataset_final = pd.merge(
    dataset,
    precip_merge,
    on="fecha",
    how="left",
    validate="one_to_one"
)

dataset_final.head()

,fecha,anio,mes,ICEN,ONI,SSN,precipitacion_mm
0,1950-01-01,1950,1,-0.75,-1.32,143.9,155.309817
1,1950-02-01,1950,2,-1.07,-1.20,134.3,164.955825
2,1950-03-01,1950,3,-1.25,-1.12,155.4,158.180623
3,1950-04-01,1950,4,-1.18,-1.08,160.6,221.687990
4,1950-05-01,1950,5,-1.22,-1.10,150.5,123.802819


In [16]:
print("Registros:", len(dataset_final))

print("\nPeriodo:")
print(
    dataset_final["fecha"].min(),
    "a",
    dataset_final["fecha"].max()
)

print("\nValores faltantes:")
print(dataset_final.isnull().sum())

print("\nDuplicados:")
print(dataset_final["fecha"].duplicated().sum())

Registros: 917

Periodo:
1950-01-01 00:00:00 a 2026-05-01 00:00:00

Valores faltantes:
fecha               0
anio                0
mes                 0
ICEN                0
ONI                 0
SSN                 0
precipitacion_mm    0
dtype: int64

Duplicados:
0


In [17]:
dataset_final.to_csv(
    "data/processed/dataset_integrado_precipitacion.csv",
    index=False,
    encoding="utf-8"
)

print("✅ Dataset de las cuatro variables creado correctamente")

✅ Dataset de las cuatro variables creado correctamente


In [18]:
!git status
!git add data/processed/dataset_integrado_precipitacion.csv

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/dataset_integrado_precipitacion.csv

nothing added to commit but untracked files present (use "git add" to track)


In [19]:
!git config user.name "gretadive"
!git config user.email "gretadive@users.noreply.github.com"

!git commit -m "Agregar dataset integrado con precipitación"

[main 751cd7c] Agregar dataset integrado con precipitación
 1 file changed, 918 insertions(+)
 create mode 100644 data/processed/dataset_integrado_precipitacion.csv


In [20]:
from google.colab import userdata
import subprocess
import base64

token = userdata.get("GITHUB_TOKEN")
usuario = "gretadive"

credenciales = base64.b64encode(
    f"{usuario}:{token}".encode()
).decode()

resultado = subprocess.run(
    [
        "git",
        "-c",
        f"http.extraHeader=Authorization: Basic {credenciales}",
        "push",
        "origin",
        "main"
    ],
    capture_output=True,
    text=True
)

if resultado.returncode == 0:
    print("✅ Dataset integrado con precipitación enviado a GitHub")
else:
    print("❌ Error:")
    print(resultado.stderr)

✅ Dataset integrado con precipitación enviado a GitHub
